In [1]:
import numpy as np
import re
import polars as pl
import scipy

from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from langchain.chains import LLMChain
from langchain_chroma import Chroma
from langchain.document_loaders import JSONLoader
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.llms import Ollama
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.documents.base import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence  # , RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)



In [2]:
path = "/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json"

In [3]:
# question = "어떤 대학교가 미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구하는지 찾아줘"
question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어"
# question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구를 한 대학 어디야?"

In [4]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            # [f"{d.metadata['seq_num']}-{d.metadata['sub_seq_num']} RANK:{i+1}\n{d.metadata['title'][:20]}:\n\n" + d.page_content for i, d in enumerate(docs)]
            [f"{d.metadata['seq_num']} RANK:{i+1}\n{d.metadata['title'][:64]}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["title"] = record.get("title")
    metadata["date"] = record.get("date")
    return metadata

loader = JSONLoader(
    file_path=path,
    jq_schema=".[]",
    content_key="content",
    text_content=True,
    metadata_func=metadata_func
)
text_splitter = RecursiveCharacterTextSplitter(
    separators="\n\n",
    chunk_size=200,
    chunk_overlap=0,
    keep_separator=True
)
embeddings = OllamaEmbeddings(
    model="tiger-gemma2"
)

In [5]:
llm = ChatOllama(
    model="tiger-gemma2",
    temperature=0.8,
    num_predict=320,
)

response = llm.invoke(question)
print(f"[{response.response_metadata['eval_duration'] / np.power(10., 9)} sec.]:\n" + "" + response.content)


[8.987735 sec.]:
미세 플라스틱이 햇빛에 노출되면 오염 줄질을 흡수하는 연구는 환경과 화학 분야에서 많은 관심을 받고 있습니다. 이러한 연구를 수행하고 있는 미국 대학은 다음과 같습니다:

1. **Stanford University (스탠포드대학교):** 스탠포드대학교의 연구자들은 미세 플라스틱이 태양광 및 자외선에 노출되어 오염 줄질을 흡수하는 방식을 조사하고 있습니다. 특히, 이러한 과정에서 생성되는 유해 화합물과 환경에 미치는 영향에 대한 연구를 진행하고 있습니다.

2. **University of California, Berkeley (캘리포니아대학교 버클리캠퍼스):** 버클리 대학교의 연구자들은 미세 플라스틱이 햇빛에 노출되면 생성되는 자유 라디칼과 환경을 오염시키는 기전에 대해 연구하고 있습니다. 또한, 이러한 과정에서 생성된 유해 물질이 생태계에 미치는 영향도 조사하고 있습니다.

3. **University of Washington (워싱턴대학교):** 워싱턴 대학교의 연구자들은 미세 플라스틱의 광분해 및 오염 줄질 흡수를 방지하는 기술 개발에 초점을 두고 있습니다. 특히, 미세 플라스


In [6]:
import yaml
from langchain.agents import create_json_agent
from langchain_community.agent_toolkits import JsonToolkit
from langchain_community.tools.json.tool import JsonSpec
from langchain_openai import OpenAI

# Chaining Test using RunnableSequence
- https://medium.com/@manoj-gupta/llm-chains-using-runnables-df500d2b7490
- https://python.langchain.com/v0.2/docs/how_to/sequence/

In [7]:
# llm = Ollama(
#     model="tiger-gemma2",
#     temperature=0.8,
#     num_predict=320,
# )
llm = ChatOllama(
    model="tiger-gemma2",
    temperature=0.8,
    num_predict=320,
)


# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

# And a query intented to prompt a language model to populate the data structure.
joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
parser = JsonOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
    template="Answer the user query.\n{query}\n",
    # template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    # partial_variables={"format_instructions": parser.get_format_instructions()},
)
prompt = PromptTemplate.from_template(
    "Answer the user query.\n{query}\n"
)

json_prompt = PromptTemplate.from_template(
    "{text} json 형식으로 출력해줘. 'answer'키만 출력해야돼",
)

# prompt = PromptTemplate.from_template

# chain = prompt | llm #| parser
# chain = prompt | llm | JsonOutputParser()

chain = prompt | llm | StrOutputParser()

# O
# chain2 = {"text": chain} | json_prompt | llm | StrOutputParser()
# O
chain2 = chain | (lambda text: {"text": text}) | json_prompt | llm | JsonOutputParser()
# O
# chain2 = RunnableSequence(
#     {
#         "text": chain,
#     },
#     json_prompt,
#     llm,
#     # StrOutputParser()
#     JsonOutputParser()
# )

# O
# chain2 = {"text": prompt | llm | StrOutputParser()} | json_prompt | llm | JsonOutputParser()

# chain = LLMChain(llm=llm, prompt=prompt) | StrOutputParser()
# chain = LLMChain(llm=llm, prompt=prompt)

# tool = chain.as_tool()
# chain2 = tool | StrOutputParser()

print(type(chain))
print(chain2)

# chain.invoke({"query": joke_query})
# tool.invoke({"query": joke_query})
# llm.invoke(joke_query)

# chain.invoke({"query": joke_query})
chain2.invoke({"query": joke_query})

NameError: name 'BaseModel' is not defined

## pipe and RunnableParallel

In [ ]:
from langchain_core.runnables import RunnableParallel

chain_with_pipe = (
    RunnableParallel(
        {"query": chain}
    ).pipe(
        prompt
    ).pipe(
        llm
    ).pipe(
        StrOutputParser()
    )
)
chain_with_pipe.invoke({"query": joke_query})

'That\'s a funny one! It seems like you\'re referencing a classic joke that plays on the double meaning of "make up." Atoms are indeed the fundamental building blocks of all matter, but in this context, "make up" also refers to telling lies or fabricating something.\n\nThe joke implies that because atoms make up everything, they could be untrustworthy since they\'re involved in creating everything around us, which can sometimes involve deception or dishonesty. It\'s a clever use of wordplay and humor to highlight the significance of atoms while poking fun at their potential for "making things up."'

In [ ]:
from pprint import pprint

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.llms import Ollama

# prompts
prompt1 = PromptTemplate.from_template(
  'What is the city {person} is from? Only respond with the name of the city.'
)
prompt2 = PromptTemplate.from_template(
  'What country is the city {city} in? Respond in {language}.'
)

# model
# model = Ollama(model="gemma:7b", temperature=0.0)
model = llm

# output parser
output_parser = StrOutputParser()

# chain
# chain = prompt1.pipe(model).pipe(output_parser) # This syntax also works
chain = prompt1 | model | output_parser
pprint(chain)

chain.invoke({"person": "Joe Biden"})

# combined chain
combined_chain = RunnableSequence(
    {
        "city": chain,
        "language": lambda inputs: inputs['language'],
        # "language": RunnablePassthrough(),
    },
    prompt2,
    model,
    output_parser
)
print("####")
pprint(combined_chain)

result = combined_chain.invoke({
  "person": "Obama",
  "language": "German",
})
print(result)

PromptTemplate(input_variables=['person'], template='What is the city {person} is from? Only respond with the name of the city.')
| Ollama(model='tiger-gemma2', num_predict=320, temperature=0.8)
| StrOutputParser()
####
{
  city: PromptTemplate(input_variables=['person'], template='What is the city {person} is from? Only respond with the name of the city.')
        | Ollama(model='tiger-gemma2', num_predict=320, temperature=0.8)
        | StrOutputParser(),
  language: RunnableLambda(...)
}
| PromptTemplate(input_variables=['city', 'language'], template='What country is the city {city} in? Respond in {language}.')
| Ollama(model='tiger-gemma2', num_predict=320, temperature=0.8)
| StrOutputParser()
Honolulu ist in den Vereinigten Staaten von Amerika, im Bundesstaat Hawaii.
